In [ ]:
from shutil import rmtree
from pathlib import Path
import os
import subprocess
import json
from pprint import pprint

from clinicaio.dataset_description import BIDSDatasetDescription, BIDSDatasetType
from clinicaio.types import SessionId, FileExtension, SubjectId, DataType, Suffix
from clinicaio.info import ImageScanInfo, SubjectInfo, SessionInfo
from clinicaio.entities import Entities
from clinicaio.dataset import BIDSDataset

# IN _env.py:
#
# from pathlib import Path
# test_bids_write_path=Path("/path/to/BIDS_out")
from _env import test_bids_write_path

rmtree(test_bids_write_path, ignore_errors=True)

dataset = BIDSDataset(
    bids_path=test_bids_write_path, 
    description=BIDSDatasetDescription(name="TEST BIDS1", version="1.10.0", dataset_type=BIDSDatasetType.RAW),
)
subject = dataset.add_subject(
    id=SubjectId("sub-ONE"),
    info=SubjectInfo(other_fields={
        "subj1": 345,
        "subj2": "texte",
        "subj3": None,
    }),
)

session = subject.add_session(
    id=SessionId("ses-ONE"),
    info=SessionInfo(
        acquisition_time="2026-04-22T01:02:03Z",
        pathology=None,
        other_fields={
            "foo": 37,
            "bar": 3929,
            "baz": "texte",
        },
    ),
)
session2 = subject.add_session(
    id=SessionId("ses-N2"),
    info=SessionInfo(
        acquisition_time="2026-04-20T11:12:13Z",
        pathology=None,
        other_fields={
            "foo": 37,
            "bar": 3929,
            "baz": "texte",
        },
    ),
)


dataset.write_to_folder(readme="TEST README BIDS")

with session.write_images() as images_writer:
    image1 = images_writer.write_image(
        data_type=DataType.PET,
        nifti_extension=FileExtension.NII_GZ,
        entities=Entities.from_str("trc-18FFDG_task-rest"),#Entities({"trc": "18FFDG", "task": "rest"}),
        suffix=Suffix("T1w"),
        scan_info=ImageScanInfo(other_fields={
            "a": "1a",
            "b": "1b",
        })
    )
    with open(image1.get_nifti_image_path(), mode="x") as f:
        print("NIFTI1", file=f)

    image2 = images_writer.write_image(
        data_type=DataType.ANAT,
        nifti_extension=FileExtension.NII_GZ,
        entities=Entities.from_str("trc-11CPIB_task-rest"),#({"trc": "11CPIB", "task": "rest"}),
        suffix=Suffix("T1w"),
        scan_info=ImageScanInfo(other_fields={
            "b": "2b",
            "c": "2c",
        })
    )
    with open(image2.get_nifti_image_path(), mode="x") as f:
        print("NIFTI2", file=f)

        
    with open(image1.get_image_companion_file_path(FileExtension.JSON), "x") as f:
        json.dump(obj={
            "name": "test",
            "size": 3092,
        }, fp=f)

# Here (when the "with" scope ends), the *_scans.tsv is automatically written from
# the scan info provided when using ImagesWriter.write_image()



read_infos = True
dataset2 = BIDSDataset.populate_from_dir(test_bids_write_path, sessions_info=read_infos, subjects_info=read_infos, image_scans_info=read_infos)
#pprint(dataset2)
print(dataset2.description)
for subject in dataset2.all_subjects():
	print(subject.id, subject.info)
	for session in subject.all_sessions():
		print("\t", session.id, session.info)
		for image in session.all_images():
			print(f"\t\t{image.data_type}: {image}")

            

subprocess.run(["tree"], check=True, cwd=test_bids_write_path)
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.read_csv(test_bids_write_path / "sub-ONE" / "ses-ONE" / "sub-ONE_ses-ONE_scans.tsv", sep="\t", dtype=str, keep_default_na=False)